# 第 54 章：Capstone 故障模式与升级处理手册

这个 notebook 用一个极小的 incident table，对比“只按紧急程度排序”的 naive baseline 和“先检查边界、扩散信号、owner 与复现状态”的故障升级 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_failure_mode_escalation_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['urgency_score'] = float(row['urgency_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} incident items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Failure signal:', ordered_counts(rows, 'failure_signal_status'))
print('Severity:', ordered_counts(rows, 'severity_status'))
print('Reproducibility:', ordered_counts(rows, 'reproducibility_status'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Escalation boundary:', ordered_counts(rows, 'escalation_boundary_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['incident_id']
    for row in rows
    if row['reference_route'] == 'local_patch_ready'
)
top4_urgency = sorted(
    rows,
    key=lambda row: row['urgency_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'local_patch_ready'
    for row in top4_urgency
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'local_patch_ready'
    for row in top4_urgency
)
baseline_ready_precision = baseline_ready_hits / len(top4_urgency)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_urgency)

print('Reference local-patch-ready items:', reference_ready_ids)
print('Top-4 by urgency:')
for row in top4_urgency:
    print(
        f"  {row['incident_id']}: urgency={row['urgency_score']:.2f}, "
        f"route={row['reference_route']}, area={row['failure_area']}"
    )
print('Baseline local-patch precision:', round(baseline_ready_precision, 3))
print('Baseline non-local intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_incident(row):
    if row['escalation_boundary_status'] in {'policy', 'privacy', 'grading'}:
        return 'escalate_to_human_review', 'incident crosses a policy, privacy, or grading boundary'
    if row['failure_signal_status'] == 'widespread':
        return 'pause_rollout_and_patch', 'failure is already spreading and rollout should pause first'
    if row['owner_status'] != 'assigned':
        return 'assign_incident_owner', 'incident has no clear owner yet'
    if row['reproducibility_status'] != 'reproduced':
        return 'collect_repro_before_escalation', 'incident still needs reproducible evidence'
    return 'local_patch_ready', 'incident is bounded, reproducible, and owned locally'


routed_rows = []
for row in rows:
    route, reason = route_incident(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Incident workflow routes:')
for row in routed_rows:
    print(
        f"  {row['incident_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
local_queue = [row for row in routed_rows if row['route'] == 'local_patch_ready']
repro_queue = [row for row in routed_rows if row['route'] == 'collect_repro_before_escalation']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_incident_owner']
pause_queue = [row for row in routed_rows if row['route'] == 'pause_rollout_and_patch']
human_queue = [row for row in routed_rows if row['route'] == 'escalate_to_human_review']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'local_patch_ready'
    for row in local_queue
) / max(len(local_queue), 1)

print('Local-patch queue:', [row['incident_id'] for row in local_queue])
print('Collect-repro queue:', [row['incident_id'] for row in repro_queue])
print('Assign-owner queue:', [row['incident_id'] for row in owner_queue])
print('Pause-rollout queue:', [row['incident_id'] for row in pause_queue])
print('Human-review queue:', [row['incident_id'] for row in human_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured local-patch precision:', round(ready_precision, 3))


In [ ]:
def incident_steps(row):
    steps = []
    if row['escalation_boundary_status'] in {'policy', 'privacy', 'grading'}:
        steps.append('先升级到课程负责人或政策复核人，不要自行拍板。')
    if row['failure_signal_status'] == 'widespread':
        steps.append('先暂停相关 rollout、入口或活动，再做补丁修复。')
    if row['owner_status'] != 'assigned':
        steps.append('指定 incident owner，并记录 first response 和 follow-up 责任。')
    if row['reproducibility_status'] != 'reproduced':
        steps.append('补复现证据：日志、截图、环境、受影响人数和 workaround。')
    return steps or ['可以本地补丁修复；保留 incident note 和修复记录。']


for row in routed_rows:
    if row['route'] != 'local_patch_ready':
        print(f"\n{row['incident_id']} -> {row['route']} ({row['failure_area']})")
        for step in incident_steps(row):
            print(' -', step)


In [ ]:
escalation_outline = [
    'Incident class: technical, grading, policy, privacy, release, or access failure',
    'Pause condition: what triggers immediate rollout pause',
    'Escalation boundary: when course staff must hand the case upward',
    'Evidence bundle: logs, screenshots, reproduction steps, and affected scope',
    'Owner chain: first response, fixer, reviewer, and outward communicator',
    'Postmortem note: what happened, what changed, and how to prevent recurrence',
]

incident_assistant_prompt = '''你是我的 capstone 故障升级助手。

任务：
1. 先阅读 incident table，不要只按 urgency 排序；
2. 先检查 escalation boundary；
3. 再检查 signal 扩散、owner 和 reproducibility；
4. 把每个事件 route 到 local_patch_ready、collect_repro_before_escalation、
   assign_incident_owner、pause_rollout_and_patch 或 escalate_to_human_review；
5. 对每个非 local 事件输出处理前 checklist。

输出格式：
- Local-patch incidents
- Collect-repro incidents
- Assign-owner incidents
- Pause-rollout incidents
- Human-review incidents
- Incident checklist by item
'''

print('Escalation playbook outline:')
for item in escalation_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(incident_assistant_prompt)


In [ ]:
incident_snapshot = {
    'dataset': DATA_PATH.name,
    'n_incidents': len(rows),
    'baseline_top4_local_patch_precision': round(baseline_ready_precision, 3),
    'baseline_non_local_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'local_patch_ready': [row['incident_id'] for row in local_queue],
    'collect_repro_before_escalation': [row['incident_id'] for row in repro_queue],
    'assign_incident_owner': [row['incident_id'] for row in owner_queue],
    'pause_rollout_and_patch': [row['incident_id'] for row in pause_queue],
    'escalate_to_human_review': [row['incident_id'] for row in human_queue],
    'python_version': platform.python_version(),
}

print('Incident escalation snapshot:')
for key, value in incident_snapshot.items():
    print(f'  {key}: {value}')
